# Temporal Decay (Frozen Schema) -- EDA

Visualises `run_temporal_decay_experiment`
(`thesis.system_eval.temporal_decay`). One schema+model is mined/fit once on
the **source window's train split** (here `source_split_mode="baseline_split"`
-- W_src = the CSCAS baseline's own train period) and **frozen**: schema,
model and decision threshold. It is then walked forward one window at a time
to the end of the timeline. `h=0` is W_src's own held-out test split; every
horizon after that is a fully external future window.

**Three feature sets** are compared head to head (all frozen, all walked the
same way):

| `feature_set` | what it is |
|---|---|
| `baseline`   | the 5 deployment-realistic base columns only |
| `symbolic`   | base + a schema mined on W_src (6 mining settings) |
| `cscas_full` | the CSCAS paper's own ~41 columns (base + SCAS + Similarity + 33 attr-similarity). A **non-deployable reference ceiling** -- SCAS/Similarity come from CSCAS's offline pipeline. `scas` is dropped for the one-class models. |
| `cscas_full_symbolic` | `cscas_full` columns + the mined `symbolic` layer (shared base columns encoded once). |

**Four models**: `logreg`, `xgboost` (supervised) and `iforest`, `ocsvm`
(one-class, Platt-scaled to an attack probability). All frozen, all walked
identically.

**Explanations.** SHAP + LIME signed importances are logged for every schema
feature at every horizon, against a background sample frozen from W_src. The
one-class models have **no analytic SHAP explainer**, so this run computes
**LIME for all four models and SHAP only for `logreg`/`xgboost`**. LIME is
therefore the common basis for any cross-model comparison.

**Novelty.** Every horizon row also carries alert-group novelty vs. W_src's
train span, at two grains: the `raw_items` token-set (`*_items`) and the
`(category, ruleset, proto)` tuple (`*_crp`). These are schema/model
independent -- one series per granularity.

Sections:
- **§1** metric decay vs. horizon -- the three schemas, per model
- **§2** decay-summary table (h=0 vs. last horizon)
- **§3** per-horizon feature-importance heatmaps (per model, per schema)
- **§4** importance **stability** vs. h=0 -- the cross-model-comparable view
- **§5** decay **rate / acceleration** vs. how much novel traffic each window brings

In [ ]:
from __future__ import annotations

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

from thesis.experiments._shared import METRIC_COLS
from thesis.paths import RESULTS_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## Config

Edit and re-run -- nothing past the loader cell needs changing.

In [ ]:
SCENARIO = "cscas"
RUN_ID = "20260909_171937_cfs"          # None -> latest run dir for SCENARIO; or "20260909_171937"

PRIMARY_METRIC = "auc"  # §1 first panel, §2 sort, §5 default rate metric
GRANULARITY = None      # None -> finest (most horizons); or pin one the run used

# §1 symbolic band: symbolic has one curve per mining setting. Show them as a
# median line + inter-quartile band instead of 6 separate lines.
SYMBOLIC_AS_BAND = True

# §3 / §4: which symbolic mining setting to use where a single schema is
# needed (heatmaps, per-schema feature lists). None -> first available.
MINING_SETTING = None

HEATMAP_TOP_K = 18      # features (rows) per heatmap, by pooled mean |importance|
STABILITY_TOP_K = 10    # top-k set size for the §4 Jaccard-overlap curve

# fixed colours so every figure agrees
FEATURE_SET_ORDER = ["baseline", "symbolic", "cscas_full", "cscas_full_symbolic"]
FEATURE_SET_COLOR = {"baseline": "#BB5566", "symbolic": "#004488",
                     "cscas_full": "#DDAA33", "cscas_full_symbolic": "#228833"}
MODEL_ORDER = ["logreg", "xgboost", "iforest", "ocsvm"]
MODEL_COLOR = dict(zip(MODEL_ORDER, ["#004488", "#66CCEE", "#228833", "#EE6677"]))

In [ ]:
sweep_dir = RESULTS_DIR / "sys-eval" / "temporal-decay" / SCENARIO
run_dirs = sorted(p for p in sweep_dir.iterdir() if p.is_dir())
run_dir = (sweep_dir / RUN_ID) if RUN_ID else run_dirs[-1]


def _read_csv_maybe_empty(path) -> pd.DataFrame:
    """A metrics-only run writes explanations.csv / lime_fidelity.csv from an
    empty frame (a bare newline) -- pd.read_csv raises EmptyDataError on
    that. Treat it as zero rows."""
    try:
        return pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


per_horizon = pd.read_csv(run_dir / "per_horizon_results.csv", low_memory=False)
decay_summary = pd.read_csv(run_dir / "decay_summary.csv")
explanations = _read_csv_maybe_empty(run_dir / "explanations.csv")
lime_fidelity = _read_csv_maybe_empty(run_dir / "lime_fidelity.csv")

NOVELTY_COLS = [c for c in per_horizon.columns
                if c.startswith(("n_novel", "frac_novel", "n_new_", "n_groups_win", "n_attack_win"))]
HAS_NOVELTY = bool(NOVELTY_COLS)
HAS_EXPLANATIONS = not explanations.empty

print(f"run: {run_dir.name}")
print(f"  per_horizon : {len(per_horizon):>6} rows")
print(f"  decay_summary: {len(decay_summary):>6} rows")
print(f"  explanations : {len(explanations):>6} rows"
      + ("" if HAS_EXPLANATIONS else "  (none -- §3/§4 will be empty)"))
print(f"  lime_fidelity: {len(lime_fidelity):>6} rows")
print()
print(f"  feature_sets : {sorted(per_horizon['feature_set'].unique())}")
print(f"  models       : {sorted(per_horizon['model'].unique())}")
print(f"  mining setts : {sorted(per_horizon['mining_setting'].dropna().unique())}")
print(f"  granularities: {sorted(per_horizon['granularity'].unique())}")
print(f"  horizons     : 0..{int(per_horizon['horizon_window_index'].max())}")
print(f"  novelty cols : {'yes' if HAS_NOVELTY else 'NO -- older run, §5 overlay disabled'}")
if HAS_EXPLANATIONS:
    cov = (explanations.groupby(['feature_set', 'model'])['method']
           .agg(lambda s: '+'.join(sorted(s.unique()))).unstack())
    print("\n  explanation methods by feature_set x model:")
    print(cov.to_string().replace("\n", "\n  "))

## Shared helpers

In [ ]:
CONFIG_COLS = ["feature_set", "mining_setting", "granularity", "model"]


def resolve_gran() -> float:
    grans = sorted(per_horizon["granularity"].unique())
    if GRANULARITY is None:
        return grans[0]
    if GRANULARITY not in grans:
        raise ValueError(f"GRANULARITY={GRANULARITY} not in run ({grans})")
    return GRANULARITY


GRAN = resolve_gran()
PH = per_horizon[per_horizon["granularity"] == GRAN].copy()
HORIZONS = sorted(PH["horizon_window_index"].unique())
FEATURE_SETS = [fs for fs in FEATURE_SET_ORDER if fs in set(PH["feature_set"])]
MODELS = [m for m in MODEL_ORDER if m in set(PH["model"])]
MINING_SETTINGS = sorted(PH.loc[PH["feature_set"] == "symbolic", "mining_setting"].dropna().unique())
MS_ONE = MINING_SETTING or (MINING_SETTINGS[0] if MINING_SETTINGS else None)


def ms_short(ms) -> str:
    """gr3_md1_mda2 -> md1/mda2 (the growth-rate prefix is frozen across the grid)."""
    if ms is None or (isinstance(ms, float) and np.isnan(ms)):
        return ""
    ms = str(ms)
    return ms[4:].replace("_", "/") if ms.startswith("gr3_") else ms.replace("_", "/")


def fs_rows(df: pd.DataFrame, feature_set: str, model: str) -> pd.DataFrame:
    """All horizon rows for one (feature_set, model) at GRAN, sorted by horizon.
    For symbolic this is 6 mining settings stacked."""
    m = df["feature_set"].eq(feature_set) & df["model"].eq(model) & df["granularity"].eq(GRAN)
    return df[m].sort_values("horizon_window_index")


def novelty_series() -> pd.DataFrame:
    """The per-horizon novelty columns (identical across every config at GRAN),
    indexed by horizon."""
    if not HAS_NOVELTY:
        return pd.DataFrame()
    cols = ["horizon_window_index", *NOVELTY_COLS]
    out = (PH[cols].drop_duplicates("horizon_window_index")
           .set_index("horizon_window_index").sort_index())
    return out


NOV = novelty_series()

## 1. Metric decay vs. horizon

One panel per model, one line per feature set, walking `h=0` (starred --
W_src's own held-out test split) to the last window. `symbolic` is drawn as
the **median over its 6 mining settings** with an inter-quartile band
(`SYMBOLIC_AS_BAND`).

The question: does the frozen `cscas_full` (paper's full feature set) hold up
over time the way the mined `symbolic` schema does, and does the reduced
`baseline` decay faster than both?

In [ ]:
def _series_for(feature_set: str, model: str, metric: str):
    """(horizons, median, q25, q75) for one (feature_set, model). For a single
    curve (baseline/cscas_full) q25==q75==median."""
    sub = fs_rows(PH, feature_set, model)
    if sub.empty or metric not in sub:
        return None
    g = sub.groupby("horizon_window_index")[metric]
    return g.median().index.values, g.median().values, g.quantile(0.25).values, g.quantile(0.75).values


def plot_decay(metric: str = None, models=None) -> None:
    metric = metric or PRIMARY_METRIC
    models = models or MODELS
    ncols = min(2, len(models))
    nrows = -(-len(models) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.4 * ncols, 4.2 * nrows), squeeze=False)
    for ax, model in zip(axes.flat, models):
        for fs in FEATURE_SETS:
            s = _series_for(fs, model, metric)
            if s is None:
                continue
            h, med, q25, q75 = s
            color = FEATURE_SET_COLOR[fs]
            ax.plot(h, med, "-o", ms=4, lw=1.8, color=color, label=fs, zorder=3)
            if SYMBOLIC_AS_BAND and (q75 > q25).any():
                ax.fill_between(h, q25, q75, color=color, alpha=0.18, lw=0, zorder=1)
            ax.scatter(h[:1], med[:1], color=color, marker="*", s=150,
                       edgecolor="black", linewidth=0.5, zorder=5)
        ax.set_title(model)
        ax.set_xlabel("horizon (window index)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)
        ax.margins(x=0.02)
    for ax in axes.flat[len(models):]:
        ax.set_visible(False)
    star = Line2D([], [], color="grey", marker="*", ms=11, ls="None",
                  markeredgecolor="black", markeredgewidth=0.5)
    h0, l0 = axes.flat[0].get_legend_handles_labels()
    fig.legend(h0 + [star], l0 + ["h=0 (W_src held-out)"],
               loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9)
    fig.suptitle(f"{SCENARIO}: {metric} decay -- frozen schema, gran={GRAN:g}")
    fig.tight_layout()


plot_decay()

In [ ]:
# Every metric, one model -- the full picture for a single detector.
def plot_all_metrics(model: str = "logreg") -> None:
    metrics = [PRIMARY_METRIC] + [m for m in METRIC_COLS if m != PRIMARY_METRIC]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), squeeze=False)
    for ax, metric in zip(axes.flat, metrics):
        for fs in FEATURE_SETS:
            s = _series_for(fs, model, metric)
            if s is None:
                continue
            h, med, q25, q75 = s
            ax.plot(h, med, "-o", ms=3.5, lw=1.6, color=FEATURE_SET_COLOR[fs], label=fs)
            if SYMBOLIC_AS_BAND and (q75 > q25).any():
                ax.fill_between(h, q25, q75, color=FEATURE_SET_COLOR[fs], alpha=0.18, lw=0)
            ax.scatter(h[:1], med[:1], color=FEATURE_SET_COLOR[fs], marker="*", s=110,
                       edgecolor="black", linewidth=0.5, zorder=5)
        ax.set_title(metric)
        ax.set_xlabel("horizon")
        ax.grid(alpha=0.25)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"{SCENARIO}: all metrics vs horizon -- model={model}, gran={GRAN:g}")
    fig.tight_layout()


plot_all_metrics("logreg")

In [ ]:
# Drill-down: symbolic broken out by mining setting (no band), one model.
def plot_symbolic_settings(model: str = "logreg", metric: str = None) -> None:
    metric = metric or PRIMARY_METRIC
    fig, ax = plt.subplots(figsize=(8, 5))
    for fs in ("baseline", "cscas_full"):
        s = _series_for(fs, model, metric)
        if s:
            ax.plot(s[0], s[1], "--", lw=2.2, color=FEATURE_SET_COLOR[fs], label=fs, zorder=4)
    pal = plt.get_cmap("viridis")
    for i, ms in enumerate(MINING_SETTINGS):
        sub = PH[(PH.feature_set == "symbolic") & (PH.model == model)
                 & (PH.mining_setting == ms)].sort_values("horizon_window_index")
        ax.plot(sub["horizon_window_index"], sub[metric], "-o", ms=3, lw=1.3,
                color=pal(i / max(1, len(MINING_SETTINGS) - 1)), label=f"sym {ms_short(ms)}")
    ax.set_xlabel("horizon (window index)")
    ax.set_ylabel(metric)
    ax.set_title(f"{SCENARIO}: {metric} -- symbolic by mining setting, model={model}")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()


plot_symbolic_settings("logreg")

## 2. Decay summary

`decay_rate_{metric} = score(h=0) - score(h_max)` (positive = the frozen model
got worse); `fpr_drift = fpr(h_max) - fpr(h=0)` (positive = more false
positives). `h_max` is the last horizon reached.

In [ ]:
sort_col = f"decay_rate_{PRIMARY_METRIC}"
ds = decay_summary.copy()
ds_view = ds.sort_values(sort_col, ascending=False) if sort_col in ds else ds
ds_view

In [ ]:
# Aggregated: median decay across mining settings, per feature_set x model.
rate_cols = [c for c in decay_summary.columns if c.startswith("decay_rate_")] + ["fpr_drift"]
agg = (decay_summary.groupby(["feature_set", "model"])[rate_cols]
       .median().round(4)
       .reindex(pd.MultiIndex.from_product([FEATURE_SETS, MODELS], names=["feature_set", "model"]))
       .dropna(how="all"))
print(f"median decay_rate / fpr_drift over mining settings (gran={GRAN:g})")
agg

## 3. Per-horizon feature-importance heatmaps

Signed mean importance per (feature, horizon): **red pushes the score toward
`attack`, blue toward `benign`**, intensity = strength. Rows are the
top-`HEATMAP_TOP_K` features by pooled mean |importance| across horizons;
columns are horizons.

Read a row left-to-right: a colour that fades or flips is a feature whose
influence on the model's output is drifting as the target window moves away
from W_src.

**Comparability.** These are model-*output* attributions, not accuracy. Raw
magnitudes are **not** comparable across model families (LinearExplainer vs
TreeExplainer vs LIME's local surrogate, plus the Platt squash for the
one-class models). Compare a row to itself over time, and compare *drift*
across models in §4 -- not the absolute colours between panels. This run has
SHAP only for `logreg`/`xgboost`; `iforest`/`ocsvm` show LIME.

In [ ]:
def _imp_pivot(feature_set, model, method, mining_setting=None):
    """feature x horizon signed-importance table for one config, or None."""
    if not HAS_EXPLANATIONS:
        return None
    e = explanations
    m = (e.feature_set.eq(feature_set) & e.model.eq(model) & e.method.eq(method)
         & e.granularity.eq(GRAN))
    if feature_set in ("symbolic", "cscas_full_symbolic"):
        m &= e.mining_setting.eq(mining_setting or MS_ONE)
    sub = e[m]
    if sub.empty:
        return None
    return sub.pivot_table(index="feature", columns="horizon_window_index",
                           values="importance", aggfunc="mean")


def _draw_heatmap(ax, pivot, title, k):
    order = pivot.abs().fillna(0).mean(axis=1).sort_values(ascending=False).head(k).index
    pivot = pivot.loc[order]
    vmax = float(np.nanmax(np.abs(pivot.values))) or 1e-9
    im = ax.imshow(np.ma.masked_invalid(pivot.values), aspect="auto",
                   cmap=plt.get_cmap("RdBu_r"),
                   norm=mcolors.TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"h{c}" for c in pivot.columns], fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=7)
    ax.set_title(title, fontsize=9)
    return im


def heatmap_grid(feature_set: str, method: str = "lime", mining_setting=None, k=None) -> None:
    """One heatmap per model for a feature set + method."""
    k = k or HEATMAP_TOP_K
    models = [m for m in MODELS if _imp_pivot(feature_set, m, method, mining_setting) is not None]
    if not models:
        print(f"no {method.upper()} importances for feature_set={feature_set}")
        return
    fig, axes = plt.subplots(1, len(models),
                             figsize=(max(4.0, 0.5 * len(HORIZONS) + 2.6) * len(models), 0.32 * k + 1.6),
                             squeeze=False)
    for ax, model in zip(axes[0], models):
        im = _draw_heatmap(ax, _imp_pivot(feature_set, model, method, mining_setting), model, k)
        ax.set_xlabel("horizon")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ms = (f"  [{ms_short(mining_setting or MS_ONE)}]"
          if feature_set in ("symbolic", "cscas_full_symbolic") else "")
    fig.suptitle(f"{SCENARIO}: {method.upper()} signed importance drift -- "
                 f"{feature_set}{ms}  (red -> attack)")
    fig.tight_layout()

In [ ]:
heatmap_grid("symbolic", "lime")

In [ ]:
heatmap_grid("symbolic", "shap")   # logreg / xgboost only

In [ ]:
heatmap_grid("cscas_full", "lime")
heatmap_grid("cscas_full_symbolic", "lime")
heatmap_grid("baseline", "lime")

## 4. Importance stability vs. h=0 (cross-model comparable)

Raw importance magnitudes don't compare across model families, but **how much
the importance vector moves** does -- every measure below is normalised to
`[-1, 1]` or `[0, 1]` regardless of model, so `logreg` and `iforest` belong
on the same axis. All are computed against each config's own **frozen h=0
importance vector** (same frozen model, same frozen schema -> the vector only
moves because the target window drifts).

| measure | 1.0 means | drop means |
|---|---|---|
| `spearman`      | same feature ranking as h=0 | ranking is reshuffling |
| `cosine`        | same signed-importance direction | direction rotating |
| `jaccard_topk`  | same top-`STABILITY_TOP_K` features | the influential set is churning |
| `signflip_frac` | *(0.0 good)* fraction of features whose sign flipped vs h=0 | features changing which class they favour |

`iforest`/`ocsvm` use **LIME** (no SHAP this run); use `method="lime"` for a
like-for-like comparison across all four models.

In [ ]:
from scipy.stats import spearmanr


def _stability_vs_h0(pivot: pd.DataFrame, k: int) -> pd.DataFrame:
    """pivot = feature x horizon signed importance. Compare every horizon to h0."""
    if pivot is None or 0 not in pivot.columns:
        return pd.DataFrame()
    v0 = pivot[0].fillna(0.0).values
    top0 = set(pd.Series(np.abs(v0), index=pivot.index).nlargest(k).index)
    rows = []
    for h in pivot.columns:
        vh = pivot[h].fillna(0.0).values
        denom = (np.linalg.norm(v0) * np.linalg.norm(vh)) or 1e-12
        toph = set(pd.Series(np.abs(vh), index=pivot.index).nlargest(k).index)
        both_nz = (v0 != 0) & (vh != 0)
        rows.append({
            "horizon_window_index": h,
            "spearman": spearmanr(v0, vh).correlation if np.ptp(v0) and np.ptp(vh) else np.nan,
            "cosine": float(v0 @ vh / denom),
            "jaccard_topk": len(top0 & toph) / len(top0 | toph),
            "signflip_frac": float(np.mean(np.sign(v0[both_nz]) != np.sign(vh[both_nz])))
                             if both_nz.any() else np.nan,
        })
    return pd.DataFrame(rows)


def stability_df(method: str = "lime") -> pd.DataFrame:
    """Long: one row per (feature_set, model, mining_setting, horizon) with the
    four stability measures vs that config's h0."""
    if not HAS_EXPLANATIONS:
        return pd.DataFrame()
    out = []
    for fs in FEATURE_SETS:
        settings = MINING_SETTINGS if fs in ("symbolic", "cscas_full_symbolic") else [None]
        for model in MODELS:
            for ms in settings:
                piv = _imp_pivot(fs, model, method, ms)
                st = _stability_vs_h0(piv, STABILITY_TOP_K)
                if st.empty:
                    continue
                st.insert(0, "feature_set", fs)
                st.insert(1, "model", model)
                st.insert(2, "mining_setting", ms)
                out.append(st)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


STAB_LIME = stability_df("lime")
STAB_LIME.head()

In [ ]:
def plot_stability(measure: str = "spearman", method_df: pd.DataFrame = None,
                   method_name: str = "LIME") -> None:
    """One panel per feature set, one line per model (median over mining
    settings for symbolic, IQR band)."""
    sdf = STAB_LIME if method_df is None else method_df
    if sdf.empty:
        print("no stability data (no explanations)")
        return
    fss = [fs for fs in FEATURE_SETS if fs in set(sdf.feature_set)]
    fig, axes = plt.subplots(1, len(fss), figsize=(5.4 * len(fss), 4.2), squeeze=False, sharey=True)
    for ax, fs in zip(axes[0], fss):
        for model in MODELS:
            g = sdf[(sdf.feature_set == fs) & (sdf.model == model)]
            if g.empty:
                continue
            gg = g.groupby("horizon_window_index")[measure]
            h = gg.median().index.values
            ax.plot(h, gg.median().values, "-o", ms=4, lw=1.7,
                    color=MODEL_COLOR[model], label=model)
            if g.mining_setting.notna().any() and g.mining_setting.nunique() > 1:
                ax.fill_between(h, gg.quantile(.25).values, gg.quantile(.75).values,
                                color=MODEL_COLOR[model], alpha=0.15, lw=0)
        ax.set_title(fs)
        ax.set_xlabel("horizon (window index)")
        ax.grid(alpha=0.25)
    axes[0][0].set_ylabel(f"{measure} vs h=0")
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f"{SCENARIO}: {method_name} importance {measure} vs h=0 "
                 f"(1.0 = unchanged{'; 0 = best' if measure == 'signflip_frac' else ''})")
    fig.tight_layout()


plot_stability("spearman")
plot_stability("cosine")
plot_stability("jaccard_topk")
plot_stability("signflip_frac")

In [ ]:
# Sanity: for the two classifiers, does SHAP tell the same stability story as LIME?
STAB_SHAP = stability_df("shap")
if not STAB_SHAP.empty:
    m = "spearman"
    both = (pd.concat([STAB_LIME.assign(expl="lime"), STAB_SHAP.assign(expl="shap")])
            .query("model in ['logreg', 'xgboost']"))
    fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(5.4 * len(FEATURE_SETS), 4), squeeze=False, sharey=True)
    for ax, fs in zip(axes[0], FEATURE_SETS):
        for model in ["logreg", "xgboost"]:
            for expl, ls in [("lime", "-"), ("shap", "--")]:
                g = both[(both.feature_set == fs) & (both.model == model) & (both.expl == expl)]
                if g.empty:
                    continue
                gg = g.groupby("horizon_window_index")[m].median()
                ax.plot(gg.index, gg.values, ls, color=MODEL_COLOR[model], lw=1.6,
                        label=f"{model} {expl}")
        ax.set_title(fs); ax.set_xlabel("horizon"); ax.grid(alpha=0.25)
    axes[0][0].set_ylabel(f"{m} vs h=0"); axes[0][0].legend(fontsize=7)
    fig.suptitle(f"{SCENARIO}: SHAP vs LIME stability agreement ({m}) -- classifiers")
    fig.tight_layout()
else:
    print("no SHAP importances logged -- skipping SHAP/LIME agreement check")

## 5. Decay rate / acceleration vs. novel traffic

Discrete derivatives of a metric along the horizon:

- **velocity** `v(h) = m(h) - m(h-1)` -- per-window change (negative = decaying)
- **acceleration** `a(h) = v(h) - v(h-1)` -- is the decay speeding up?

Overlaid (right axis) with how much genuinely-unseen traffic each window
brings: `n_new_item_types` (raw_items token-sets not present in W_src's train
span) and `frac_novel_items_attack` (share of that window's *attacks* whose
type is new). Horizon `h` is a window index -- the windows differ in size and
composition, so these are per-window rates, not per-unit-time; second
differences on ~10 points are noisy, read them for direction not magnitude.

In [ ]:
def decay_dynamics(metric: str = None) -> pd.DataFrame:
    """Per (feature_set, model, mining_setting): velocity + acceleration of
    `metric` along the horizon."""
    metric = metric or PRIMARY_METRIC
    keys = ["feature_set", "model", "mining_setting"]
    out = []
    for k, g in PH.groupby(keys, dropna=False):
        g = g.sort_values("horizon_window_index")
        v = g[metric].diff()
        out.append(g[["horizon_window_index"]].assign(
            **dict(zip(keys, k if isinstance(k, tuple) else (k,))),
            value=g[metric].values, velocity=v.values, acceleration=v.diff().values))
    return pd.concat(out, ignore_index=True)


DYN = decay_dynamics()
DYN.head()

In [ ]:
def plot_rate_vs_novelty(metric: str = None, kind: str = "velocity",
                         novelty_col: str = "n_new_item_types") -> None:
    metric = metric or PRIMARY_METRIC
    dyn = decay_dynamics(metric)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), squeeze=False)
    for ax, model in zip(axes.flat, MODELS):
        for fs in FEATURE_SETS:
            g = dyn[(dyn.feature_set == fs) & (dyn.model == model)]
            if g.empty:
                continue
            gg = g.groupby("horizon_window_index")[kind]
            ax.plot(gg.median().index, gg.median().values, "-o", ms=4, lw=1.7,
                    color=FEATURE_SET_COLOR[fs], label=fs)
            if g.mining_setting.nunique() > 1:
                ax.fill_between(gg.median().index, gg.quantile(.25).values,
                                gg.quantile(.75).values, color=FEATURE_SET_COLOR[fs],
                                alpha=0.15, lw=0)
        ax.axhline(0, color="grey", lw=0.8)
        ax.set_title(model)
        ax.set_xlabel("horizon (window index)")
        ax.set_ylabel(f"{kind} of {metric}")
        ax.grid(alpha=0.25)
        if HAS_NOVELTY and novelty_col in NOV:
            ax2 = ax.twinx()
            ax2.bar(NOV.index, NOV[novelty_col], color="grey", alpha=0.18, width=0.7, zorder=0)
            ax2.set_ylabel(novelty_col, color="grey")
            ax2.tick_params(axis="y", colors="grey")
    axes.flat[0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{SCENARIO}: {kind} of {metric} vs horizon  "
                 f"(bars = {novelty_col}, right axis)")
    fig.tight_layout()


plot_rate_vs_novelty(kind="velocity")
plot_rate_vs_novelty(kind="acceleration")

In [ ]:
# Does the decay rate track novelty? Pool every (config, horizon>=1) point:
# |velocity| of the metric vs the window's novelty, coloured by feature set.
if HAS_NOVELTY:
    from scipy.stats import spearmanr
    metric = PRIMARY_METRIC
    dyn = decay_dynamics(metric).dropna(subset=["velocity"])
    dyn = dyn.merge(NOV.reset_index(), on="horizon_window_index", how="left")
    dyn["abs_velocity"] = dyn["velocity"].abs()

    xcol = "n_new_item_types"
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), squeeze=False)
    for ax, ycol in zip(axes[0], ["abs_velocity", "acceleration"]):
        d = dyn.dropna(subset=[ycol, xcol])
        for fs in FEATURE_SETS:
            f = d[d.feature_set == fs]
            ax.scatter(f[xcol], f[ycol].abs() if ycol == "acceleration" else f[ycol],
                       s=22, alpha=0.6, color=FEATURE_SET_COLOR[fs], label=fs)
        rho, p = spearmanr(d[xcol], d[ycol].abs())
        ax.set_xlabel(xcol)
        ax.set_ylabel(f"|{ycol}| of {metric}")
        ax.set_title(f"Spearman rho = {rho:.2f}  (p = {p:.3f},  n = {len(d)})")
        ax.grid(alpha=0.25)
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f"{SCENARIO}: does the frozen model decay faster where more traffic is novel?")
    fig.tight_layout()
else:
    print("no novelty columns in this run -- rerun the experiment to get the overlay")

In [ ]:
# LIME local fidelity (R^2 of the local linear surrogate) over the horizon --
# a drop means the decision surface is getting harder to approximate locally,
# independent of whether the signed weights are drifting (§4).
if not lime_fidelity.empty:
    lf = lime_fidelity[lime_fidelity.granularity == GRAN]
    fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(5.4 * len(FEATURE_SETS), 4),
                             squeeze=False, sharey=True)
    for ax, fs in zip(axes[0], FEATURE_SETS):
        for model in MODELS:
            g = lf[(lf.feature_set == fs) & (lf.model == model)]
            if g.empty:
                continue
            gg = g.groupby("horizon_window_index")["mean_fidelity"].median()
            ax.plot(gg.index, gg.values, "-o", ms=4, color=MODEL_COLOR[model], label=model)
        ax.set_title(fs); ax.set_xlabel("horizon"); ax.grid(alpha=0.25)
    axes[0][0].set_ylabel("LIME mean local R^2"); axes[0][0].legend(fontsize=8)
    fig.suptitle(f"{SCENARIO}: LIME local fidelity vs horizon")
    fig.tight_layout()
else:
    print("no lime_fidelity rows")